# Rare Disease Detection with Machine Learning  
## A beginner-friendly healthcare notebook with synthetic toy data

This notebook teaches a **simple healthcare machine learning workflow** for detecting a **rare disease**.

We will use **synthetic toy data**, which means the data are fake and created only for learning.

---

## What you will learn

In this notebook, we will go step by step through:

1. creating a rare disease dataset  
2. understanding **class imbalance**  
3. training a simple machine learning model  
4. evaluating the model with healthcare-friendly metrics  
5. checking **calibration**  
6. doing simple **threshold tuning**  
7. understanding **data leakage**  
8. doing simple **external validation**  
9. checking fairness across patient groups  
10. looking at model interpretability  
11. thinking about the cost of false positives and false negatives  
12. writing a final clinical-style insight

---

## Important idea

In rare disease detection, **accuracy alone is not enough**.

Why?

Because if only 2% of patients have the disease, a model can predict "no disease" for almost everyone and still look accurate.

That is why healthcare machine learning often focuses on:

- **recall / sensitivity**
- **precision**
- **specificity**
- **negative predictive value**
- **AUC**
- **PR AUC**
- **calibration**
- **clinical threshold choice**

We will explain each one simply.

## Clinical scenario

Imagine we are screening patients for a **rare disease**.

Each row is one patient.  
The target variable is:

- `rare_disease = 1` → patient truly has the rare disease  
- `rare_disease = 0` → patient does not have the rare disease  

We will create simple patient features such as:

- age
- symptom score
- lab score
- imaging score
- inflammation marker
- a patient group variable for fairness analysis

This is **not real patient data**.

## Step 0: Setup

We first import the libraries we need.

We keep the code simple and use common Python tools:

- `numpy` for numbers
- `pandas` for tables
- `matplotlib` and `seaborn` for plots
- `scikit-learn` for machine learning

In [ ]:
# STEP 0: IMPORT LIBRARIES

try:
    import google.colab
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    !pip -q install numpy pandas matplotlib seaborn scikit-learn

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    roc_curve,
    precision_recall_curve,
    brier_score_loss
)
from sklearn.calibration import calibration_curve
from sklearn.inspection import permutation_importance

np.random.seed(42)
sns.set(style="whitegrid")
pd.set_option("display.max_columns", 50)

print("Setup complete. Running in Colab:", IN_COLAB)

## Why we use Logistic Regression in this notebook

We use **Logistic Regression** because it is:

- simple
- common in healthcare
- easier to explain than many complex models
- good for predicting probabilities

A more complex model is not always better for learning.

## Step 1: Create synthetic toy data

We will create a simple fake healthcare dataset.

### Goal

We want the rare disease to be:

- **uncommon**
- a little more likely when symptom, lab, and imaging values are worse
- slightly different across patient groups so we can study fairness

### Important note

This is still a **toy example**.  
Real healthcare data are more messy and more complicated.

In [ ]:
# STEP 1: CREATE SYNTHETIC TOY DATA

n = 4000

# Create a simple patient group variable for fairness analysis.
# Group A and Group B are just toy labels for learning.
group = np.random.choice(["Group A", "Group B"], size=n, p=[0.65, 0.35])

# Create simple patient features.
age = np.random.normal(loc=50, scale=15, size=n).clip(18, 90)
symptom_score = np.random.normal(loc=5, scale=2, size=n).clip(0, 10)
lab_score = np.random.normal(loc=100, scale=20, size=n).clip(40, 180)
imaging_score = np.random.normal(loc=3, scale=1.5, size=n).clip(0, 8)
inflammation_marker = np.random.normal(loc=4, scale=2, size=n).clip(0, 15)

# Build a simple hidden risk score.
# Larger symptom score, imaging score, inflammation, and older age slightly increase risk.
# Lower lab_score increases risk in this toy example, so we subtract part of it.
risk_score = (
    -7.2
    + 0.025 * age
    + 0.35 * symptom_score
    - 0.018 * lab_score
    + 0.45 * imaging_score
    + 0.20 * inflammation_marker
)

# Add a small group effect just to create a fairness discussion example.
risk_score = np.where(group == "Group B", risk_score + 0.25, risk_score)

# Convert risk score into probability using the logistic formula.
disease_probability = 1 / (1 + np.exp(-risk_score))

# Create the rare disease outcome.
rare_disease = np.random.binomial(1, disease_probability)

# Build the dataset.
df = pd.DataFrame({
    "age": age,
    "symptom_score": symptom_score,
    "lab_score": lab_score,
    "imaging_score": imaging_score,
    "inflammation_marker": inflammation_marker,
    "group": group,
    "rare_disease": rare_disease
})

df.head()

## Step 2: Inspect the data

Before training a model, always inspect the data first.

We want to check:

- number of rows and columns
- data types
- summary statistics
- how rare the disease is

In [ ]:
# STEP 2: BASIC INSPECTION

print("Rows, Columns:", df.shape)

print("\nData types:")
print(df.dtypes)

print("\nSummary statistics:")
display(df.describe(include="all").T)

In [ ]:
# STEP 2B: CHECK HOW RARE THE DISEASE IS

class_counts = df["rare_disease"].value_counts().sort_index()
class_percent = df["rare_disease"].value_counts(normalize=True).sort_index() * 100

print("Class counts:")
print(class_counts)

print("\nClass percentages:")
print(class_percent.round(2))

rare_rate = df["rare_disease"].mean() * 100
print(f"\nRare disease rate: {rare_rate:.2f}%")

In [ ]:
# STEP 2C: PLOT THE CLASS DISTRIBUTION

plt.figure(figsize=(6, 4))
sns.countplot(data=df, x="rare_disease")
plt.title("Rare Disease Class Distribution")
plt.xlabel("Rare disease (0 = no, 1 = yes)")
plt.ylabel("Number of patients")
plt.show()

## Why class imbalance matters

If the disease is very rare, a model can look "good" by predicting **no disease** most of the time.

That is why healthcare data scientists do **not** trust accuracy alone.

We need a wider set of metrics.

## Step 3: Prepare data for machine learning

We need:

- features `X`
- target `y`

We also need to turn the `group` text column into numbers so the model can use it.

We will use **one-hot encoding**, which is a simple way to turn categories into columns.

In [ ]:
# STEP 3: PREPARE FEATURES AND TARGET

# Turn the group text column into numeric columns.
df_model = pd.get_dummies(df, columns=["group"], drop_first=True)

# Features
X = df_model.drop(columns=["rare_disease"])

# Target
y = df_model["rare_disease"]

print("Feature columns:")
print(list(X.columns))

## Step 4: Split into train, test, and external validation sets

We will create:

- **training set**: used to fit the model
- **test set**: used to evaluate the model
- **external validation set**: used as a simple stand-in for a new outside dataset

In real healthcare work, external validation should come from a **different hospital, time period, or population**.

Here we simulate that idea in a simple way.

In [ ]:
# STEP 4: SPLIT THE DATA

# First split off an external validation set.
X_temp, X_external, y_temp, y_external = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

# Then split the remaining data into training and test sets.
X_train, X_test, y_train, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.25,
    stratify=y_temp,
    random_state=42
)

print("Training set shape:", X_train.shape)
print("Test set shape:", X_test.shape)
print("External validation set shape:", X_external.shape)

print("\nRare disease rate in training set:", round(y_train.mean() * 100, 2), "%")
print("Rare disease rate in test set:", round(y_test.mean() * 100, 2), "%")
print("Rare disease rate in external validation set:", round(y_external.mean() * 100, 2), "%")

## Step 5: Train a simple baseline model

We now train a **Logistic Regression** model.

This model gives us probabilities between 0 and 1.

That is useful in healthcare because:

- we often care about risk probability
- we may choose different clinical thresholds later

In [ ]:
# STEP 5: TRAIN A BASELINE MODEL

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

# Predicted probability of disease
test_prob = model.predict_proba(X_test)[:, 1]

# Default class prediction using threshold = 0.50
test_pred = (test_prob >= 0.50).astype(int)

print("Model training complete.")

## Step 6: Build a healthcare-friendly evaluation function

We will calculate many metrics that are commonly used in health-related prediction tasks.

### Metrics we will include

- Accuracy
- Precision
- Recall (Sensitivity)
- Specificity
- F1-score
- NPV (Negative Predictive Value)
- ROC AUC
- PR AUC
- Brier score

### Why so many metrics?

Because each metric tells a different part of the story.

In [ ]:
# STEP 6: EVALUATION FUNCTION

def evaluate_model(y_true, y_prob, threshold=0.50):
    y_pred = (y_prob >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)   # sensitivity
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    f1 = f1_score(y_true, y_pred, zero_division=0)
    npv = tn / (tn + fn) if (tn + fn) > 0 else 0
    roc_auc = roc_auc_score(y_true, y_prob)
    pr_auc = average_precision_score(y_true, y_prob)
    brier = brier_score_loss(y_true, y_prob)

    results = pd.DataFrame({
        "Metric": [
            "Accuracy",
            "Precision",
            "Recall (Sensitivity)",
            "Specificity",
            "F1-score",
            "Negative Predictive Value",
            "ROC AUC",
            "PR AUC",
            "Brier Score"
        ],
        "Value": [
            accuracy,
            precision,
            recall,
            specificity,
            f1,
            npv,
            roc_auc,
            pr_auc,
            brier
        ]
    })

    return results, (tn, fp, fn, tp), y_pred

In [ ]:
# STEP 6B: EVALUATE THE MODEL ON THE TEST SET

results_test, cm_test, y_pred_test = evaluate_model(y_test, test_prob, threshold=0.50)

display(results_test)

tn, fp, fn, tp = cm_test
print("Confusion matrix values:")
print("TN =", tn)
print("FP =", fp)
print("FN =", fn)
print("TP =", tp)

## How to read these healthcare metrics

### Accuracy
How often the model is correct overall.

### Precision
Among patients predicted as disease-positive, how many truly have the disease?

### Recall / Sensitivity
Among patients who truly have the disease, how many did the model catch?

### Specificity
Among patients who do not have the disease, how many did the model correctly call negative?

### Negative Predictive Value (NPV)
Among patients predicted as disease-negative, how many are truly negative?

### F1-score
A balance between precision and recall.

### ROC AUC
How well the model separates positive and negative cases across many thresholds.

### PR AUC
Very useful for rare disease tasks because it focuses more on the positive class.

### Brier score
Measures how good the predicted probabilities are. Lower is better.

In [ ]:
# STEP 6C: PLOT THE CONFUSION MATRIX

cm = confusion_matrix(y_test, y_pred_test)

plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title("Confusion Matrix on Test Set")
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.show()

In [ ]:
# STEP 6D: PLOT ROC CURVE

fpr, tpr, _ = roc_curve(y_test, test_prob)

plt.figure(figsize=(6, 4))
plt.plot(fpr, tpr, label=f"Model ROC AUC = {roc_auc_score(y_test, test_prob):.3f}")
plt.plot([0, 1], [0, 1], linestyle="--", label="Random model")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.show()

In [ ]:
# STEP 6E: PLOT PRECISION-RECALL CURVE

precision_vals, recall_vals, _ = precision_recall_curve(y_test, test_prob)

plt.figure(figsize=(6, 4))
plt.plot(recall_vals, precision_vals, label=f"Model PR AUC = {average_precision_score(y_test, test_prob):.3f}")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve")
plt.legend()
plt.show()

## Step 7: Calibration

Calibration asks:

> When the model says a patient has 20% risk, is the true event rate really close to 20%?

This is important in healthcare because risk probabilities are often used for:

- screening decisions
- referral decisions
- treatment discussions

A model can rank patients well but still give poorly calibrated probabilities.

In [ ]:
# STEP 7: CALIBRATION CURVE

prob_true, prob_pred = calibration_curve(y_test, test_prob, n_bins=10)

plt.figure(figsize=(6, 4))
plt.plot(prob_pred, prob_true, marker="o", label="Model")
plt.plot([0, 1], [0, 1], linestyle="--", label="Perfect calibration")
plt.xlabel("Predicted probability")
plt.ylabel("Observed frequency")
plt.title("Calibration Curve")
plt.legend()
plt.show()

print("Brier score on test set:", round(brier_score_loss(y_test, test_prob), 4))

## Step 8: Threshold tuning

By default, many models use a threshold of **0.50**.

But in healthcare, we may choose a different threshold depending on the clinical goal.

For example:

- if missing a rare disease is dangerous, we may choose a **lower threshold**
- that usually increases recall but may also increase false positives

We will compare several thresholds.

In [ ]:
# STEP 8: COMPARE DIFFERENT THRESHOLDS

threshold_list = [0.10, 0.20, 0.30, 0.40, 0.50]

all_threshold_results = []

for th in threshold_list:
    results_th, cm_th, _ = evaluate_model(y_test, test_prob, threshold=th)
    small = results_th.set_index("Metric")["Value"].to_dict()
    all_threshold_results.append({
        "Threshold": th,
        "Recall": small["Recall (Sensitivity)"],
        "Specificity": small["Specificity"],
        "Precision": small["Precision"],
        "NPV": small["Negative Predictive Value"],
        "F1-score": small["F1-score"]
    })

threshold_df = pd.DataFrame(all_threshold_results)
display(threshold_df)

In [ ]:
# STEP 8B: CHOOSE A SIMPLE CLINICAL OPERATING POINT

# Example clinical rule:
# We want recall to be at least 0.80 if possible,
# because missing a rare disease is costly.

possible_rows = threshold_df[threshold_df["Recall"] >= 0.80]

if len(possible_rows) > 0:
    chosen_threshold = possible_rows.sort_values(by="Specificity", ascending=False).iloc[0]["Threshold"]
else:
    chosen_threshold = 0.50

print("Chosen threshold for a simple clinical goal:", chosen_threshold)

In [ ]:
# STEP 8C: EVALUATE THE CHOSEN THRESHOLD

results_chosen, cm_chosen, y_pred_chosen = evaluate_model(y_test, test_prob, threshold=chosen_threshold)

display(results_chosen)

tn, fp, fn, tp = cm_chosen
print("Confusion matrix at chosen threshold:")
print("TN =", tn, "FP =", fp, "FN =", fn, "TP =", tp)

## Why clinically chosen operating points matter

A model is not only "good" or "bad".

A model can behave differently depending on the chosen threshold.

In healthcare, the threshold should depend on the clinical context, for example:

- screening test
- triage tool
- confirmatory diagnostic support
- very costly treatment decision

Different tasks need different trade-offs.

## Step 9: External validation

Now we test the same model on the **external validation set**.

In real healthcare work, external validation means testing on data that are more truly outside the training setting.

Why do this?

Because a model that works only in one dataset may fail in a new hospital or new patient population.

In [ ]:
# STEP 9: EXTERNAL VALIDATION

external_prob = model.predict_proba(X_external)[:, 1]
results_external, cm_external, y_pred_external = evaluate_model(y_external, external_prob, threshold=chosen_threshold)

display(results_external)

tn, fp, fn, tp = cm_external
print("External validation confusion matrix values:")
print("TN =", tn)
print("FP =", fp)
print("FN =", fn)
print("TP =", tp)

## Step 10: Data leakage

Data leakage means the model accidentally gets information that would not be available in real life.

This can make the model look unrealistically good.

Examples of leakage in healthcare:

- using a future diagnosis code
- using a lab collected after the outcome happened
- using a discharge variable to predict an event that happened during admission

We will show a tiny toy example.

In [ ]:
# STEP 10: TOY EXAMPLE OF DATA LEAKAGE

df_leak = df_model.copy()

# Create a fake leakage feature.
# This feature is strongly tied to the outcome and should NOT exist in a real prediction setting.
df_leak["leak_feature"] = df_leak["rare_disease"] + np.random.normal(0, 0.05, size=len(df_leak))

X_leak = df_leak.drop(columns=["rare_disease"])
y_leak = df_leak["rare_disease"]

X_train_leak, X_test_leak, y_train_leak, y_test_leak = train_test_split(
    X_leak, y_leak,
    test_size=0.20,
    stratify=y_leak,
    random_state=42
)

leak_model = LogisticRegression(max_iter=1000)
leak_model.fit(X_train_leak, y_train_leak)
leak_prob = leak_model.predict_proba(X_test_leak)[:, 1]

results_leak, _, _ = evaluate_model(y_test_leak, leak_prob, threshold=0.50)

display(results_leak)

## Leakage lesson

If the leakage model looks *much better* than the earlier model, that is a warning sign.

A model can seem amazing simply because it was given future or unrealistic information.

In healthcare ML, leakage is one of the most important mistakes to avoid.

## Step 11: Fairness across patient groups

We will now check whether model performance is similar for:

- Group A
- Group B

This is important because a model can work well overall but poorly for one patient group.

In [ ]:
# STEP 11: FAIRNESS CHECK BY GROUP

# Recover the original group label for the test set.
group_test = df.loc[X_test.index, "group"]

fairness_rows = []

for g in ["Group A", "Group B"]:
    mask = group_test == g
    y_true_g = y_test[mask]
    y_prob_g = test_prob[mask]

    if len(y_true_g) > 0 and y_true_g.nunique() > 1:
        results_g, _, _ = evaluate_model(y_true_g, y_prob_g, threshold=chosen_threshold)
        small_g = results_g.set_index("Metric")["Value"].to_dict()

        fairness_rows.append({
            "Group": g,
            "Accuracy": small_g["Accuracy"],
            "Precision": small_g["Precision"],
            "Recall": small_g["Recall (Sensitivity)"],
            "Specificity": small_g["Specificity"],
            "ROC AUC": small_g["ROC AUC"],
            "PR AUC": small_g["PR AUC"]
        })

fairness_df = pd.DataFrame(fairness_rows)
display(fairness_df)

## Fairness lesson

If one group has much lower recall or much lower specificity, that matters.

For example:

- lower recall in one group means more missed rare disease cases in that group
- lower specificity in one group means more false alarms in that group

Fairness in healthcare is not just a technical issue.  
It can become a patient safety and equity issue.

## Step 12: Interpretability

Interpretability means trying to understand **why** the model is making predictions.

Since we used Logistic Regression, we can look at:

1. model coefficients  
2. permutation importance

This will not give perfect causal truth, but it helps us understand which features matter more to the model.

In [ ]:
# STEP 12A: LOOK AT MODEL COEFFICIENTS

coef_df = pd.DataFrame({
    "Feature": X_train.columns,
    "Coefficient": model.coef_[0]
}).sort_values(by="Coefficient", ascending=False)

display(coef_df)

### How to read the coefficients

- a **positive** coefficient means larger values of that feature push the model toward predicting disease
- a **negative** coefficient means larger values push the model away from predicting disease

Because this is logistic regression, the coefficients affect the **log-odds**, not the raw probability directly.

For beginners, it is enough to think:

- positive = increases predicted risk
- negative = decreases predicted risk

In [ ]:
# STEP 12B: PERMUTATION IMPORTANCE

perm = permutation_importance(
    model,
    X_test,
    y_test,
    n_repeats=10,
    random_state=42
)

perm_df = pd.DataFrame({
    "Feature": X_test.columns,
    "Importance": perm.importances_mean
}).sort_values(by="Importance", ascending=False)

display(perm_df)

In [ ]:
# STEP 12C: PLOT PERMUTATION IMPORTANCE

plt.figure(figsize=(8, 5))
sns.barplot(data=perm_df, x="Importance", y="Feature")
plt.title("Permutation Importance")
plt.show()

## Step 13: Cost of false positives vs false negatives

In healthcare, the two main mistakes are:

### False positive
The patient does not have the disease, but the model says they do.

Possible consequences:
- extra tests
- extra clinic visits
- patient anxiety
- unnecessary cost

### False negative
The patient truly has the disease, but the model says they do not.

Possible consequences:
- missed diagnosis
- delayed treatment
- worse patient outcomes

Often, false negatives are more costly in rare disease detection.

In [ ]:
# STEP 13: SIMPLE COST EXAMPLE

# We create a toy cost rule.
# Example:
# false positive cost = 1 unit
# false negative cost = 10 units

false_positive_cost = 1
false_negative_cost = 10

tn, fp, fn, tp = cm_chosen

total_cost = fp * false_positive_cost + fn * false_negative_cost

print("Chosen threshold:", chosen_threshold)
print("False positives:", fp)
print("False negatives:", fn)
print("Total toy clinical cost:", total_cost)

## Why cost-based thinking matters

A model with slightly lower accuracy can still be better clinically if it avoids many dangerous false negatives.

That is why healthcare model evaluation should connect to:

- patient harm
- workflow burden
- treatment delay
- downstream testing cost

## Step 14: Final model summary

Now we will put our main findings into a simple final table.

In [ ]:
# STEP 14: SUMMARY TABLE

summary_table = pd.DataFrame({
    "Setting": ["Default threshold on test set", "Chosen clinical threshold on test set", "External validation"],
    "Threshold": [0.50, chosen_threshold, chosen_threshold],
    "Recall": [
        evaluate_model(y_test, test_prob, 0.50)[0].set_index("Metric").loc["Recall (Sensitivity)", "Value"],
        evaluate_model(y_test, test_prob, chosen_threshold)[0].set_index("Metric").loc["Recall (Sensitivity)", "Value"],
        evaluate_model(y_external, external_prob, chosen_threshold)[0].set_index("Metric").loc["Recall (Sensitivity)", "Value"]
    ],
    "Specificity": [
        evaluate_model(y_test, test_prob, 0.50)[0].set_index("Metric").loc["Specificity", "Value"],
        evaluate_model(y_test, test_prob, chosen_threshold)[0].set_index("Metric").loc["Specificity", "Value"],
        evaluate_model(y_external, external_prob, chosen_threshold)[0].set_index("Metric").loc["Specificity", "Value"]
    ],
    "Precision": [
        evaluate_model(y_test, test_prob, 0.50)[0].set_index("Metric").loc["Precision", "Value"],
        evaluate_model(y_test, test_prob, chosen_threshold)[0].set_index("Metric").loc["Precision", "Value"],
        evaluate_model(y_external, external_prob, chosen_threshold)[0].set_index("Metric").loc["Precision", "Value"]
    ],
    "ROC AUC": [
        evaluate_model(y_test, test_prob, 0.50)[0].set_index("Metric").loc["ROC AUC", "Value"],
        evaluate_model(y_test, test_prob, chosen_threshold)[0].set_index("Metric").loc["ROC AUC", "Value"],
        evaluate_model(y_external, external_prob, chosen_threshold)[0].set_index("Metric").loc["ROC AUC", "Value"]
    ],
    "PR AUC": [
        evaluate_model(y_test, test_prob, 0.50)[0].set_index("Metric").loc["PR AUC", "Value"],
        evaluate_model(y_test, test_prob, chosen_threshold)[0].set_index("Metric").loc["PR AUC", "Value"],
        evaluate_model(y_external, external_prob, chosen_threshold)[0].set_index("Metric").loc["PR AUC", "Value"]
    ]
})

display(summary_table.round(3))

## Step 15: Final insight

A beginner-friendly healthcare interpretation might be:

- The model can separate low-risk and high-risk patients reasonably well.
- Accuracy alone is not enough because the disease is rare.
- Recall, precision, PR AUC, and calibration are very important.
- Lowering the threshold can improve recall, which may be useful when missing disease is dangerous.
- External validation helps check whether the model still works on new data.
- Leakage can create unrealistically high performance.
- Fairness should be checked across patient groups.
- Interpretability helps us understand what the model is using.
- Clinical model choice should depend on the cost of false positives versus false negatives.

In real healthcare projects, the next steps would usually include:

- better data quality checks
- more realistic external validation
- threshold selection with clinicians
- calibration improvement if needed
- subgroup safety review
- monitoring after deployment

## Practice ideas for you

Try changing these and rerun the notebook:

1. make the disease even rarer
2. make the threshold lower or higher
3. change the false negative cost
4. make the two groups more different
5. remove one feature and see what happens

That is how you learn patterns instead of memorizing code.

In [ ]:
# OPTIONAL PRACTICE CELL

# Example ideas:
# - change the threshold
# - retrain with fewer features
# - compare the summary again

print("Practice here.")